# Aula-Exercício: Plataforma Semi-Submersível

## 1. Configuração

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

In [ ]:
Lp = 30        # meio comprimento do pontoon
H = 20         # calado
Dc = 10        # diâmetro da coluna
Dp = 10        # diâmetro do pontoon
GM = 1.56      # altura metacêntrica longitudinal e transversal
rho = 1025     # densidade da água
g = 9.81       # aceleração da gravidade
Rg = 0.5 * Lp  # raio de giro para inércia própria
alfa = 90.0001 * np.pi / 180  # direção de onda
zetah = 0.02   # amortecimento heave
zetar = 0.05   # amortecimento roll
zetap = 0.05   # amortecimento pitch
CM = 1         # coeficiente de massa adicional

Sc = np.pi * (Dc / 2) ** 2  # área seccional de cada coluna
AWF = 4 * Sc                # área de linha d'água
Vc = 4 * Sc * H             # volume deslocado total das colunas

Sp = np.pi * (Dp / 2) ** 2  # área seccional de cada pontoon
Vp = 4 * Sp * (2 * Lp)      # volume deslocado total dos pontoons

VT = Vp + Vc
Massa_plat = rho * VT
I_plat = Massa_plat * (Rg**2)

## 2. Massas e Inércias Adicionais — Períodos Naturais

In [ ]:
Mah = CM * rho * Vp
wnh = np.sqrt(rho * g * AWF / (Massa_plat + Mah))

Iaa = rho * CM * Vp * 2 / 3 * (Lp**2)
wna = np.sqrt(Massa_plat * g * GM / (I_plat + Iaa))

print(f'wn_heave     = {wnh:.2f} rad/s')
print(f'Tn_heave     = {2 * np.pi / wnh:.2f} s')
print(f'wn_roll,pitch = {wna:.2f} rad/s')
print(f'Tn_roll,pitch = {2 * np.pi / wna:.2f} s')

## 3. Vetor de Frequências

Relação de dispersão em águas profundas: $k = \omega^2 / g$.

In [ ]:
w = np.linspace(2 * np.pi / 600, 2 * np.pi / 4, 1000)
k = (w**2) / g

## 4. Grau de Liberdade: Heave

In [ ]:
A_onda = 1  # amplitude unitária para adimensionalização

# Força nos pontoons
F = -(1 + CM) * rho * Vp * (w**2) * A_onda * np.exp(-k * (H - Dp / 2))
Q1 = 0.5 * (
    np.sin(k * Lp * np.cos(alfa)) * np.cos(k * Lp * np.sin(alfa))
    / (k * Lp * np.cos(alfa))
    + np.sin(k * Lp * np.sin(alfa)) * np.cos(k * Lp * np.cos(alfa))
    / (k * Lp * np.sin(alfa))
)

# Força nas colunas
Fc = rho * g * AWF * A_onda * np.exp(-k * H)
G1 = np.cos(k * (Lp + Dc / 2) * np.sin(alfa)) * np.cos(
    k * (Lp + Dc / 2) * np.cos(alfa)
)

# Força total e RAO
Fres_heave = Fc * G1 + F * Q1
Betah = w / wnh
H_heave = np.abs(Fres_heave) / (
    rho * g * AWF * np.sqrt((1 - Betah**2) ** 2 + (2 * zetah * Betah) ** 2)
)

In [ ]:
plt.figure(figsize=(10, 6), dpi=100, facecolor='w', edgecolor='k')
plt.subplot(2, 1, 1)
plt.plot(w, np.abs(Fres_heave), 'b')
plt.xlabel('w [rad/s]')
plt.ylabel('Força de heave [N/m]')
plt.grid()
plt.subplot(2, 1, 2)
plt.plot(w, H_heave, 'b')
plt.xlabel('w [rad/s]')
plt.ylabel('RAO de heave [m/m]')
plt.plot([wnh, wnh], [0, 2.5], 'r--')
plt.grid()
plt.tight_layout()
plt.show()

## 5. Grau de Liberdade: Roll

In [ ]:
delta = 0
Betaa = w / wna

Q2 = 0.5 * (
    np.sin(k * Lp * np.sin(alfa - delta))
    * np.sin(k * Lp * np.cos(alfa - delta))
    / (k * Lp * np.cos(alfa - delta))
    - np.cos(k * Lp * np.cos(alfa - delta))
    / (k * Lp * np.sin(alfa - delta))
    * (
        np.cos(k * Lp * np.sin(alfa - delta))
        - np.sin(k * Lp * np.sin(alfa - delta)) / (k * Lp * np.sin(alfa - delta))
    )
)
G2 = np.sin(k * (Lp + Dc / 2) * np.sin(alfa - delta)) * np.cos(
    k * (Lp + Dc / 2) * np.cos(alfa - delta)
)

Nres_roll = Fc * Lp * G2 + F * Lp * Q2
H_roll = np.abs(Nres_roll) / (
    Massa_plat * g * GM * np.sqrt((1 - Betaa**2) ** 2 + (2 * zetar * Betaa) ** 2)
)

In [ ]:
plt.figure(figsize=(10, 6), dpi=100, facecolor='w', edgecolor='k')
plt.subplot(2, 1, 1)
plt.plot(w, np.abs(Nres_roll), 'k')
plt.xlabel('w [rad/s]')
plt.ylabel('Momento de Roll [N·m/m]')
plt.grid()
plt.subplot(2, 1, 2)
plt.plot(w, H_roll * 180 / np.pi, 'k')
plt.xlabel('w [rad/s]')
plt.ylabel('RAO de Roll [graus/m]')
plt.plot([wna, wna], [0, 5], 'r--')
plt.grid()
plt.tight_layout()
plt.show()

## 6. Grau de Liberdade: Pitch

In [ ]:
delta = np.pi / 2

Q2 = 0.5 * (
    np.sin(k * Lp * np.sin(alfa - delta))
    * np.sin(k * Lp * np.cos(alfa - delta))
    / (k * Lp * np.cos(alfa - delta))
    - np.cos(k * Lp * np.cos(alfa - delta))
    / (k * Lp * np.sin(alfa - delta))
    * (
        np.cos(k * Lp * np.sin(alfa - delta))
        - np.sin(k * Lp * np.sin(alfa - delta)) / (k * Lp * np.sin(alfa - delta))
    )
)
G2 = np.sin(k * (Lp + Dc / 2) * np.sin(alfa - delta)) * np.cos(
    k * (Lp + Dc / 2) * np.cos(alfa - delta)
)

Nres_pitch = Fc * Lp * G2 + F * Lp * Q2
H_pitch = np.abs(Nres_pitch) / (
    Massa_plat * g * GM * np.sqrt((1 - Betaa**2) ** 2 + (2 * zetap * Betaa) ** 2)
)

In [ ]:
plt.figure(figsize=(10, 6), dpi=100, facecolor='w', edgecolor='k')
plt.subplot(2, 1, 1)
plt.plot(w, np.abs(Nres_pitch), 'b')
plt.xlabel('w [rad/s]')
plt.ylabel('Momento de Pitch [N·m/m]')
plt.grid()
plt.subplot(2, 1, 2)
plt.plot(w, H_pitch * 180 / np.pi, 'b')
plt.xlabel('w [rad/s]')
plt.ylabel('RAO de Pitch [graus/m]')
plt.plot([wna, wna], [0, 5], 'r--')
plt.grid()
plt.tight_layout()
plt.show()

## 7. Curvas em Função do Período

In [ ]:
plt.figure(figsize=(10, 10), dpi=100, facecolor='w', edgecolor='k')
plt.subplot(3, 1, 1)
plt.plot(2 * np.pi / w, H_heave, 'r')
plt.xlabel('T [s]')
plt.ylabel('RAO de heave [m/m]')
plt.xlim((0, 100))
plt.grid()
plt.subplot(3, 1, 2)
plt.plot(2 * np.pi / w, H_roll * 180 / np.pi, 'k')
plt.xlabel('T [s]')
plt.ylabel('RAO de roll [graus/m]')
plt.xlim((0, 100))
plt.grid()
plt.subplot(3, 1, 3)
plt.plot(2 * np.pi / w, H_pitch * 180 / np.pi, 'b')
plt.xlabel('T [s]')
plt.ylabel('RAO de pitch [graus/m]')
plt.xlim((0, 100))
plt.grid()
plt.tight_layout()
plt.show()

## Apêndice — Limite Assintótico

$$\frac{\sin(k L_p \sin\alpha) \cos(k L_p \cos\alpha)}{k L_p \sin\alpha}$$

In [ ]:
x = np.linspace(0.0001, 0.1, 1000)

yy = np.sin(np.sin(x)) * np.cos(np.cos(x)) / np.sin(x)
plt.plot(x, yy)
plt.show()

[np.cos(np.cos(0)), yy[0]]